# Definining Cohort

**The goal of this notebook is to identify patients with advanced urothelial who received first-line treatment with pembrolizumab or carboplatin-based chemotherapy.**

In [1]:
import numpy as np
import pandas as pd

In [2]:
# Function that returns number of rows and count of unique PatientIDs for a dataframe. 
def row_ID(dataframe):
    row = dataframe.shape[0]
    ID = dataframe['PatientID'].nunique()
    return row, ID

## 1. Pembrolizumab

In [3]:
therapy = pd.read_csv('../data/LineOfTherapy.csv')

In [4]:
therapy.query('LineNumber == 1').IsMaintenanceTherapy.value_counts(dropna = False).head(20)

IsMaintenanceTherapy
False    9382
True      363
Name: count, dtype: int64

In [5]:
therapy.query('LineNumber == 1').query('IsMaintenanceTherapy == True').LineName.value_counts()

LineName
Avelumab    363
Name: count, dtype: int64

In [6]:
(
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    .LineName.value_counts()
    .head(10)
)

LineName
Carboplatin,Gemcitabine    2226
Cisplatin,Gemcitabine      2068
Pembrolizumab              1486
Atezolizumab                681
Carboplatin,Paclitaxel      320
Nivolumab                   301
Gemcitabine                 278
MVAC                        261
Clinical Study Drug         252
Cisplatin                   137
Name: count, dtype: int64

In [7]:
pembro_df = (
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    .query('LineName == "Pembrolizumab"')
    [['PatientID', 'LineName', 'StartDate']]
    .assign(LineName = 'pembro'))

In [8]:
pembro_df.sample(3)

,PatientID,LineName,StartDate
10252,FAA3048AE3D59,pembro,2020-05-08
12525,F50E1B5D5B44C,pembro,2020-03-10
5487,F800C3AF4ABC0,pembro,2023-03-03


In [9]:
row_ID(pembro_df)

(1486, 1486)

## 2. Carboplatin-based chemotherapy

In [10]:
carbo_df = (
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == False')
    .query('LineName == "Carboplatin,Gemcitabine"')
    [['PatientID', 'LineName', 'StartDate']]
    .assign(LineName = 'carbo'))

In [11]:
carbo_df.sample(3)

,PatientID,LineName,StartDate
16195,FE6032C2CA63E,carbo,2017-01-06
16798,F010A6FCB19E6,carbo,2011-02-09
6304,F0BFE14D8C499,carbo,2014-10-29


In [12]:
row_ID(carbo_df)

(2226, 2226)

## 3. Received avelumab maintenance

In [13]:
avelumab_maintenance_ids = (
    therapy
    .query('LineNumber == 1')
    .query('IsMaintenanceTherapy == True')
    .query('LineName == "Avelumab"')
    .PatientID
)

## 4. Combine dataframes and export to csv 

In [14]:
firstline_pembro_carbo_index = pd.concat([pembro_df, carbo_df], axis = 0)

In [15]:
row_ID(firstline_pembro_carbo_index)

(3712, 3712)

In [16]:
firstline_pembro_carbo_index['avelumab_maintenance'] = np.where(firstline_pembro_carbo_index.PatientID.isin(avelumab_maintenance_ids), 1, 0)

In [17]:
firstline_pembro_carbo_index.sample(3)

,PatientID,LineName,StartDate,avelumab_maintenance
7202,F6A57C8641455,carbo,2017-10-23,0
7295,FC785244F20B7,carbo,2018-11-12,0
8421,F32D9166F7FC8,carbo,2019-08-16,0


In [18]:
firstline_pembro_carbo_index.LineName.value_counts()

LineName
carbo     2226
pembro    1486
Name: count, dtype: int64

In [19]:
firstline_pembro_carbo_index.to_csv('../outputs/pembro_carbo_index.csv', index = False)